# IBKR Overnight Study — 15:55 → Next Open

Models the IBKR minute-bar panel: **504 sessions × 149 tickers**, roughly double the Bloomberg
sample, with a true 15:55 entry price and `late_ret` (last-30-minute return) — a feature Bloomberg
data structurally could not provide.

## What is being tested

Two feature sets, run side by side:

| Set | Features | Question |
|---|---|---|
| **full** | 19 price/volume/vol features + `late_ret` | Does `late_ret` add to what Bloomberg already had? |
| **minimal** | `intraday_ret`, `late_ret` only | Does the new signal stand alone, undiluted? |

Both are also run with earnings windows excluded. The comparison is the point: if `full` reproduces
the Bloomberg IC of ~0.023 and adding `late_ret` lifts it, that is a real finding. If neither moves,
the last half-hour carries nothing and the line of enquiry closes.

## What the Bloomberg run established

| Result | Value |
|---|---|
| Best IC (sector-neutral, ex-earnings) | 0.0226, t = 2.74 |
| Best gross top-minus-bottom | 6.4 bp vs 12 bp round trip |
| Held-out last 126 sessions | IC −0.0015, t = −0.11 |

The holdout collapse is the finding to beat. Both segments were walk-forward out-of-sample, so it
was decay or regime shift, not overfitting. **The same holdout discipline is applied here**, and it
is the number that decides whether anything real is happening.

## Honest priors

The 15:55 entry differs from the close by ~19 bp on a typical day but correlates 0.989 with it, and
averaged slightly *worse* (+2.97 vs +3.63 bp). So the entry-price change alone is unlikely to
rescue the strategy. `late_ret` is the genuine unknown.

In [ ]:
# ============================ Cell 1 — Config and data
import warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

CFG = {
    "panel":     "ib_out/panel_research.parquet",   # SEALED: holdout not reachable
    "earnings":  "earnings_dates.csv",
    "sectors":   "sectors.csv",        # optional: ticker,sector
    "target":    "target_1555_to_open",   # or target_close_to_open
    "earnings_blackout_days": 2,       # revision_date can lag the print by a day or two
    "min_train_sessions": 160,
    "test_sessions":      None,        # None -> everything after min_train
    "refit_every_days":   21,
    "holdout_sessions":   63,          # inner check; the real holdout is sealed on disk
    "winsor":  (0.005, 0.995),
    "top_decile_frac": 0.10,
    "round_trip_cost_bp": 12.0,
    "min_coverage": 0.60,
    "min_names_per_session": 40,
}

panel = pd.read_parquet(CFG["panel"])
panel["session"] = pd.to_datetime(panel["session"]).dt.normalize()
panel = panel.sort_values(["ticker", "session"]).reset_index(drop=True)
print(f"panel   : {panel.shape}  {panel['ticker'].nunique()} tickers, "
      f"{panel['session'].nunique()} sessions")
print(f"          {panel['session'].min().date()} -> {panel['session'].max().date()}")
print(f"columns : {panel.columns.tolist()}")

if "SEALED" in os.listdir("ib_out") and "research" in CFG["panel"]:
    print("SEALED    : reading research split only; holdout is on disk and unread")

earn = pd.read_csv(CFG["earnings"], parse_dates=["ann_date"])
earn["ticker"] = earn["ticker"].astype(str).str.split().str[0]
earn = earn[earn["ticker"].isin(panel["ticker"].unique())]
print(f"earnings: {len(earn):,} dates, {earn['ticker'].nunique()} of "
      f"{panel['ticker'].nunique()} panel tickers")

if os.path.exists(CFG["sectors"]):
    sect = pd.read_csv(CFG["sectors"])
    sect.columns = [c.lower() for c in sect.columns]
    sect["ticker"] = sect["ticker"].astype(str).str.split().str[0]
    panel = panel.merge(sect[["ticker", "sector"]], on="ticker", how="left")
    HAVE_SECTOR = panel["sector"].notna().mean() > 0.8
    print(f"sectors : {panel['sector'].nunique()} sectors, "
          f"{panel['sector'].notna().mean():.1%} coverage")
else:
    HAVE_SECTOR = False
    print("sectors : sectors.csv not found -- sector-neutral variants SKIPPED.")
    print("          Sector-neutralisation lifted the Bloomberg t-stat from 1.90 to 2.71,")
    print("          so it is worth exporting. In BQuant:")
    print("            s = bq.execute(bql.Request(members, {'sector': bq.data.gics_sector_name()}))")
    print("            d = list(s)[0].df().reset_index(); d.columns=['ticker','sector']")
    print("            d['ticker']=d['ticker'].str.split().str[0]; d.to_csv('sectors.csv',index=False)")

In [ ]:
# ============================ Cell 2 — Does the earnings flag actually align?
# ann_date comes from is_eps revision_date, which is when the quarter's EPS was
# first reported. That may lag the actual print by a day or two, so verify the
# flag lands on the big moves before trusting it as a filter.
print("=" * 68)
print("EARNINGS ALIGNMENT CHECK")
print("=" * 68)

bl = CFG["earnings_blackout_days"]
spans = [earn.assign(session=earn["ann_date"] + pd.Timedelta(days=k))[["ticker", "session"]]
         for k in range(-bl, bl + 1)]
flag = pd.concat(spans, ignore_index=True).drop_duplicates()
flag["is_earnings"] = True

panel = panel.merge(flag, on=["ticker", "session"], how="left")
panel["is_earnings"] = panel["is_earnings"].fillna(False).astype(bool)

tgt = CFG["target"]
p = panel.dropna(subset=[tgt])
share = p["is_earnings"].mean()
print(f"flagged stock-sessions : {int(p['is_earnings'].sum()):,} of {len(p):,} ({share:.2%})")
print(f"expected if aligned    : ~{(2*bl+1)/63:.2%}  (a {2*bl+1}-day window each quarter)")

for lab, sub in [("earnings window", p[p["is_earnings"]]), ("other", p[~p["is_earnings"]])]:
    print(f"\n{lab:<16} n={len(sub):>7,}  |target| mean={sub[tgt].abs().mean()*1e4:>7.1f} bp"
          f"  sd={sub[tgt].std()*1e4:>7.1f} bp"
          f"  P(|move|>5%)={ (sub[tgt].abs()>0.05).mean():.3%}")

ratio = (p.loc[p['is_earnings'], tgt].std() / p.loc[~p['is_earnings'], tgt].std())
print(f"\nvolatility ratio (earnings / other) = {ratio:.2f}")
if ratio > 1.5:
    print("GOOD: the flag lands on genuinely volatile sessions -- usable as a filter.")
else:
    print("WARNING: flagged sessions are not much more volatile than the rest.")
    print("         ann_date may be misaligned; try raising earnings_blackout_days.")

big = p[p[tgt].abs() > 0.10]
print(f"\nmoves >10%: {len(big):,}, of which {big['is_earnings'].mean():.1%} are flagged")

In [ ]:
# ============================ Cell 3 — Features
print("=" * 68)
print("FEATURES")
print("=" * 68)

df = panel.copy()
open_col = "px_open_adj" if "px_open_adj" in df.columns else "px_open"
print(f"open column: {open_col}"
      f"{'  (dividend-adjusted)' if open_col=='px_open_adj' else '  (NOT dividend-adjusted)'}")

g = df.groupby("ticker")
df["intraday_ret"]   = df["entry_price"] / df[open_col] - 1.0
df["prev_close"]     = g["px_close"].shift(1)
df["prev_overnight"] = df[open_col] / df["prev_close"] - 1.0
df["ret_1d"]  = g["entry_price"].pct_change(1, fill_method=None)
df["ret_5d"]  = g["entry_price"].pct_change(5, fill_method=None)
df["ret_20d"] = g["entry_price"].pct_change(20, fill_method=None)
df["ma20_gap"] = df["entry_price"] / g["entry_price"].transform(
    lambda s: s.rolling(20, min_periods=15).mean()) - 1.0
df["rv_10d"] = g["ret_1d"].transform(lambda s: s.rolling(10, min_periods=8).std()) * np.sqrt(252)
df["rv_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=15).std()) * np.sqrt(252)
df["on_mean_20"] = g["prev_overnight"].transform(lambda s: s.rolling(20, min_periods=15).mean())

rng = (df["high_to_entry"] - df["low_to_entry"]).replace(0, np.nan)
df["range_pos"] = (df["entry_price"] - df["low_to_entry"]) / rng
df["true_range"] = rng / df["prev_close"]
df["vol_ratio"] = df["vol_to_entry"] / g["vol_to_entry"].transform(
    lambda s: s.rolling(20, min_periods=15).mean()).replace(0, np.nan)
df["log_dollar_vol"] = np.log((df["entry_price"] * df["vol_to_entry"])
                              .replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

# No index in the IBKR panel: the equal-weighted cross-section IS the market.
for c in ["intraday_ret", "ret_1d", "ret_5d", "rv_20d", "late_ret"]:
    df[f"market_{c}"] = df.groupby("session")[c].transform("mean")
df["relative_intraday"] = df["intraday_ret"] - df["market_intraday_ret"]
df["relative_5d"]       = df["ret_5d"] - df["market_ret_5d"]
df["relative_late"]     = df["late_ret"] - df["market_late_ret"]

for c in ["intraday_ret", "late_ret", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_20d"]:
    df[f"rank_{c}"] = df.groupby("session")[c].rank(pct=True) - 0.5

FEATURE_SETS = {
    "full": ["intraday_ret", "late_ret", "prev_overnight", "on_mean_20",
             "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_10d", "rv_20d",
             "range_pos", "true_range", "vol_ratio", "log_dollar_vol",
             "market_intraday_ret", "market_late_ret", "relative_intraday",
             "relative_5d", "relative_late",
             "rank_intraday_ret", "rank_late_ret", "rank_ret_5d", "rank_rv_20d"],
    "full_no_late": ["intraday_ret", "prev_overnight", "on_mean_20",
                     "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_10d", "rv_20d",
                     "range_pos", "true_range", "vol_ratio", "log_dollar_vol",
                     "market_intraday_ret", "relative_intraday", "relative_5d",
                     "rank_intraday_ret", "rank_ret_5d", "rank_rv_20d"],
    "minimal": ["intraday_ret", "late_ret"],
}

df = df.replace([np.inf, -np.inf], np.nan)
tgt = CFG["target"]
df["y"] = df[tgt] - df.groupby("session")[tgt].transform("mean")   # train on cross-sectional
print(f"\ntarget: {tgt} (session-demeaned for training, raw for P&L)")

for nm, cols in FEATURE_SETS.items():
    cov = df[cols].notna().mean()
    weak = cov[cov < CFG["min_coverage"]]
    print(f"{nm:<14} {len(cols):>2} features, min coverage {cov.min():.3f}"
          f"{'  PRUNING ' + str(list(weak.index)) if len(weak) else ''}")
    FEATURE_SETS[nm] = [c for c in cols if cov.get(c, 0) >= CFG["min_coverage"]]

In [ ]:
# ============================ Cell 4 — Walk-forward engine
def make_model():
    return HistGradientBoostingRegressor(
        loss="squared_error", learning_rate=0.045, max_iter=180,
        max_leaf_nodes=15, min_samples_leaf=80, l2_regularization=2.0,
        random_state=42)


def walk_forward(data, features, label, verbose=True):
    d = data.dropna(subset=list(features) + ["y", CFG["target"]]).copy()
    sess = np.array(sorted(d["session"].unique()))
    n_test = CFG["test_sessions"] or (len(sess) - CFG["min_train_sessions"])
    test_days = sess[-n_test:]

    parts, mdl, last = [], None, -10**9
    for i, day in enumerate(test_days):
        tr = d[d["session"] < day]
        te = d[d["session"] == day]
        if tr["session"].nunique() < CFG["min_train_sessions"] or len(te) < CFG["min_names_per_session"]:
            continue
        if mdl is None or (i - last) >= CFG["refit_every_days"]:
            lo, hi = tr["y"].quantile(CFG["winsor"])
            mdl = make_model().fit(tr[features], tr["y"].clip(lo, hi))
            last = i
            if verbose:
                print(f"   [{label}] refit {i+1:>4}/{len(test_days)} @ "
                      f"{pd.Timestamp(day).date()} ({tr['session'].nunique()} train)", flush=True)
        te = te.copy()
        te["pred"] = mdl.predict(te[features])
        parts.append(te[["session", "ticker", CFG["target"], "y", "is_earnings", "pred"]])
    if not parts:
        return None
    return pd.concat(parts, ignore_index=True)


def daily_stats(oos, pred_col="pred"):
    rows = []
    frac = CFG["top_decile_frac"]
    for day, dd in oos.groupby("session", sort=True):
        if len(dd) < CFG["min_names_per_session"]:
            continue
        n = max(2, int(len(dd) * frac))
        rows.append({
            "session": day,
            "ic": dd[pred_col].corr(dd[CFG["target"]], method="spearman"),
            "top": dd.nlargest(n, pred_col)[CFG["target"]].mean(),
            "bot": dd.nsmallest(n, pred_col)[CFG["target"]].mean()})
    dd = pd.DataFrame(rows)
    dd["ls"] = dd["top"] - dd["bot"]
    return dd


def summarise(dd, label):
    n = len(dd); sd = dd["ic"].std(); pos = (dd["ic"] > 0).sum()
    return pd.Series({
        "sessions": n,
        "mean_IC": dd["ic"].mean(),
        "IC_sd": sd,
        "IC_t": dd["ic"].mean() / sd * np.sqrt(n) if sd else np.nan,
        "pos_rate": pos / n,
        "sign_z": (pos - n/2) / np.sqrt(n/4),
        "gross_bp": dd["ls"].mean() * 1e4,
        "net_bp": dd["ls"].mean() * 1e4 - CFG["round_trip_cost_bp"],
        "ls_t": dd["ls"].mean() / dd["ls"].std() * np.sqrt(n) if dd["ls"].std() else np.nan,
    }, name=label)

In [ ]:
# ============================ Cell 5 — Run all variants
print("=" * 68)
print("WALK-FORWARD")
print("=" * 68)

results, dailies = {}, {}
for nm in ["full", "full_no_late", "minimal"]:
    oos = walk_forward(df, FEATURE_SETS[nm], nm)
    if oos is None:
        print(f"{nm}: no predictions"); continue
    results[nm] = oos
    dailies[nm] = daily_stats(oos)
    clean = oos[~oos["is_earnings"]]
    if len(clean) > 0.5 * len(oos):
        dailies[nm + " (ex-earn)"] = daily_stats(clean)

table = pd.DataFrame([summarise(v, k) for k, v in dailies.items()])
print()
display(table.round(4))

null_sd = 1.0 / np.sqrt(results[list(results)[0]].groupby("session").size().mean())
print(f"IC sd under the null = {null_sd:.4f}. Much above that is factor structure, not noise.")
print(f"round-trip cost assumed: {CFG['round_trip_cost_bp']} bp")

if "full" in dailies and "full_no_late" in dailies:
    a, b = table.loc["full", "mean_IC"], table.loc["full_no_late", "mean_IC"]
    print(f"\nlate_ret contribution: IC {b:+.4f} -> {a:+.4f}  ({(a-b)*1e4:+.1f} bp of IC)")
    print("   The whole point of the IBKR download is whether this difference is real.")

In [ ]:
# ============================ Cell 6 — Deciles, holdout, power
best = table["IC_t"].astype(float).idxmax()
bd = dailies[best]
print(f"BEST VARIANT: {best}\n")

base = best.replace(" (ex-earn)", "")
oos = results[base]
if "ex-earn" in best:
    oos = oos[~oos["is_earnings"]]

tmp = oos.copy()
tmp["decile"] = tmp.groupby("session")["pred"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 10, labels=False, duplicates="drop") + 1)
dec = tmp.groupby("decile")[CFG["target"]].agg(
    mean_bp=lambda s: s.mean()*1e4, n="size").round(2)
print("Decile monotonicity:"); display(dec)
if 1 in dec.index and 10 in dec.index:
    print(f"D10-D1 = {dec.loc[10,'mean_bp']-dec.loc[1,'mean_bp']:.2f} bp gross")
print("A real signal steps across buckets, not just at the extremes.\n")

ic_sd, n_obs = bd["ic"].std(), len(bd)
print(f"POWER: IC sd {ic_sd:.4f} over {n_obs} sessions")
for t in (0.01, 0.015, 0.02, 0.03):
    need = int(np.ceil((2*ic_sd/t)**2))
    print(f"   true IC {t:.3f} at t=2 -> "
          f"{'detectable' if n_obs >= need else f'NOT detectable (need ~{need})'}")

hs = CFG["holdout_sessions"]
print(f"\nHOLDOUT — last {hs} sessions, never used for any decision:")
print(f"{'variant':<26} {'tune IC':>9} {'tune t':>7} {'HOLD IC':>9} {'HOLD t':>7} "
      f"{'HOLD bp':>8}")
for k, dd in dailies.items():
    if len(dd) <= hs + 60:
        continue
    cut = dd["session"].sort_values().iloc[-hs]
    tu, ho = dd[dd["session"] < cut], dd[dd["session"] >= cut]
    tt = tu["ic"].mean()/tu["ic"].std()*np.sqrt(len(tu)) if tu["ic"].std() else np.nan
    ht = ho["ic"].mean()/ho["ic"].std()*np.sqrt(len(ho)) if ho["ic"].std() else np.nan
    print(f"{k:<26} {tu['ic'].mean():>+9.4f} {tt:>+7.2f} {ho['ic'].mean():>+9.4f} "
          f"{ht:>+7.2f} {ho['ls'].mean()*1e4:>+8.2f}")
print("\nA holdout that collapses versus the tuning window means the signal is not durable.")
print("The Bloomberg run went IC +0.036 (t=3.47) -> -0.002 (t=-0.11). Beat that.")

In [ ]:
# ============================ Cell 7 — Charts
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
for k, dd in dailies.items():
    ax[0].plot(dd["session"], (dd["ls"].fillna(0)+1).cumprod()-1, lw=1.1, label=k)
ax[0].axhline(0, color="k", lw=0.8)
if len(bd) > CFG["holdout_sessions"]:
    ax[0].axvline(bd["session"].sort_values().iloc[-CFG["holdout_sessions"]],
                  color="red", ls="--", lw=1, label="holdout starts")
ax[0].set_title("Cumulative top-minus-bottom, gross of costs")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].plot(bd["session"], bd["ic"].rolling(63).mean(), lw=1.2)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title(f"Rank IC, 63-day rolling mean — {best}")
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

stamp = pd.Timestamp.today().strftime("%Y%m%d")
table.to_csv(f"ib_results_{stamp}.csv")
results[base].to_csv(f"ib_oos_{stamp}.csv", index=False)
print(f"wrote ib_results_{stamp}.csv and ib_oos_{stamp}.csv")